In [ ]:
from cemp_software_settings import load_and_apply_settings
result = load_and_apply_settings()
print(result)

In [ ]:
import os
import subprocess
import re
import csv
from concurrent.futures import ThreadPoolExecutor
from openbabel import openbabel
from openpyxl import Workbook
import time
import os
import subprocess
import re
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
from openbabel import openbabel
from openpyxl import Workbook
from openbabel import pybel
import shutil
import time
import pandas as pd
import glob
from typing import Dict, List  
from rdkit import Chem        
import os                     
import json                   
import re                     

In [ ]:

sobtop_directory = result["sobtop_home"]
print(sobtop_directory)

In [ ]:



def run_sobtop(mol2_path, chg_path, top_path, itp_path):
    
    
    commands = f'7\n10\n{chg_path}\n0\n1\n2\n4\n{top_path}\n{itp_path}\n0\n'
    
    
    subprocess.run(['cd', sobtop_directory], shell=True)
    
    
    
    process = subprocess.Popen(
        ['./sobtop', mol2_path],
        cwd=sobtop_directory,
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    
    stdout, stderr = process.communicate(commands)

    
    print(stdout)
    
    
    if stderr:
        print(stderr)

In [ ]:

def create_polymer_itp_top():
    for filename in os.listdir('.'):
        if filename.endswith('.pdb') and filename.startswith('polymer'):
            
            base_filename = os.path.splitext(filename)[0]
            print(base_filename)
            
            mol2_path = os.path.abspath(base_filename + '.mol2')
            chg_path = os.path.abspath(base_filename + '.chg')
            top_path = os.path.abspath(base_filename + '.top')
            itp_path = os.path.abspath(base_filename + '.itp')
            
            run_sobtop(mol2_path, chg_path, top_path, itp_path)


    
    pattern = re.compile(r"^\[ moleculetype \]\s*\n;\s*name\s+nrexcl\s*\npolymer_(\S+)\s+(\d+)", re.MULTILINE)
    
    
    for filename in os.listdir('.'):
        
        if filename.endswith('.itp') and filename.startswith('polymer'):
            
            with open(filename, 'r') as file:
                content = file.read()
    
            
            new_content = pattern.sub(r"[ moleculetype ]\n; name          nrexcl\n\1       \2", content)
    
            
            if new_content != content:
                with open(filename, 'w') as file:
                    file.write(new_content)
    
    print("All matching .itp files have been processed.")


In [ ]:
def main():
    
    df = pd.read_excel('System_homopolymer.xlsx')

    
    polymer_name = df['Name'].astype(str).tolist()
    
    
    for name in polymer_name:
        
        if name in df['Name'].values:
            print(f"--------{name}-------{name}----------{name}--------{name}---------{name}-------{name}----------{name}--------{name}--------")
            
            create_polymer_itp_top()
            print(f"--------{name}-------{name}----------{name}--------{name}---------{name}-------{name}----------{name}--------{name}--------")


In [ ]:
if __name__ == '__main__':
    main()

In [ ]:

import pandas as pd

def ensure_copolymer_name(df: pd.DataFrame) -> pd.DataFrame:
    
    
    if 'copolymer_name' not in df.columns:
        
        first_name = df['Name'].iloc[0]
        
        fill_value = f"polymer_{first_name}"
        
        df['copolymer_name'] = fill_value
    
    return df

In [ ]:
df = pd.read_excel('System_homopolymer.xlsx')

df = df.dropna(subset=['Name', 'SMILES']).copy()
for _column in ['Name', 'SMILES']:
    _text_values = df[_column].astype(str).str.strip()
    df = df[(_text_values != '') & (_text_values.str.lower() != 'nan')].copy()
if df.empty:
    raise ValueError("Public message removed for release.")
df = df.reset_index(drop=True)
ensure_copolymer_name(df)
df.to_excel('System_homopolymer.xlsx', index=None)

In [ ]:
def load_polymer_atom_list(file_path: str) -> List[str]:
    
    
    if not os.path.isfile(file_path):
        raise FileNotFoundError("Public message removed for release.")

    
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    
    if not isinstance(data, list):
        raise ValueError("Public message removed for release.")

    
    if not all(isinstance(x, str) for x in data):
        raise ValueError("Public message removed for release.")

    
    return data

In [ ]:


def generate_lowest_energy_conformer_openbabel(smiles: str, num_confs: int = 50, forcefield: str = "UFF"):
    
    
    obConversion = openbabel.OBConversion()
    obConversion.SetInFormat("smi")
    
    
    obmol = openbabel.OBMol()
    obConversion.ReadString(obmol, smiles)
    
    
    obmol.AddHydrogens()
    
    
    builder = openbabel.OBBuilder()
    builder.Build(obmol)
    
    
    cs = openbabel.OBConformerSearch()
    cs.Setup(obmol, num_confs, True)
    cs.Search()
    cs.GetConformers(obmol)
    
    
    forcefield = forcefield.upper()
    if forcefield == "MMFF94":
        ff = openbabel.OBForceField.FindForceField("mmff94")
    elif forcefield == "UFF":
        ff = openbabel.OBForceField.FindForceField("uff")
    else:
        raise ValueError("Public message removed for release.")
    
    lowest_energy = float('inf')
    lowest_conf_index = None
    nconfs = obmol.NumConformers()
    
    
    for i in range(nconfs):
        obmol.SetConformer(i)
        if not ff.Setup(obmol):
            print("Public message removed for release.")
            continue
        ff.ConjugateGradients(250, 1.0e-4)
        ff.GetCoordinates(obmol)
        energy = ff.Energy()
        if energy < lowest_energy:
            lowest_energy = energy
            lowest_conf_index = i

    
    if lowest_conf_index is None:
        return None, None, None
    
    atomic_number_to_symbol = {
        1: "H",    2: "He",   3: "Li",   4: "Be",   5: "B",    6: "C",    7: "N",    8: "O",    9: "F",    10: "Ne",
        11: "Na",  12: "Mg",  13: "Al",  14: "Si",  15: "P",   16: "S",   17: "Cl",  18: "Ar",  19: "K",   20: "Ca",
        21: "Sc",  22: "Ti",  23: "V",   24: "Cr",  25: "Mn",  26: "Fe",  27: "Co",  28: "Ni",  29: "Cu",  30: "Zn",
        31: "Ga",  32: "Ge",  33: "As",  34: "Se",  35: "Br",  36: "Kr",  37: "Rb",  38: "Sr",  39: "Y",   40: "Zr",
        41: "Nb",  42: "Mo",  43: "Tc",  44: "Ru",  45: "Rh",  46: "Pd",  47: "Ag",  48: "Cd",  49: "In",  50: "Sn",
        51: "Sb",  52: "Te",  53: "I",   54: "Xe",  55: "Cs",  56: "Ba",  57: "La",  58: "Ce",  59: "Pr",  60: "Nd",
        61: "Pm",  62: "Sm",  63: "Eu",  64: "Gd",  65: "Tb",  66: "Dy",  67: "Ho",  68: "Er",  69: "Tm",  70: "Yb",
        71: "Lu",  72: "Hf",  73: "Ta",  74: "W",   75: "Re",  76: "Os",  77: "Ir",  78: "Pt",  79: "Au",  80: "Hg",
        81: "Tl",  82: "Pb",  83: "Bi",  84: "Po",  85: "At",  86: "Rn",  87: "Fr",  88: "Ra",  89: "Ac",  90: "Th",
        91: "Pa",  92: "U",   93: "Np",  94: "Pu",  95: "Am",  96: "Cm",  97: "Bk",  98: "Cf",  99: "Es", 100: "Fm",
        101: "Md", 102: "No", 103: "Lr", 104: "Rf", 105: "Db", 106: "Sg", 107: "Bh", 108: "Hs", 109: "Mt", 110: "Ds",
        111: "Rg", 112: "Cn", 113: "Nh", 114: "Fl", 115: "Mc", 116: "Lv", 117: "Ts", 118: "Og"
    }

    
    obmol.SetConformer(lowest_conf_index)
    coordinates = []
    
    for atom in pybel.ob.OBMolAtomIter(obmol):
        atomic_num = atom.GetAtomicNum()
        symbol = atomic_number_to_symbol.get(atomic_num, str(atomic_num))  
        x = atom.GetX()
        y = atom.GetY()
        z = atom.GetZ()
        coordinates.append((symbol, x, y, z))
    
    return lowest_conf_index, lowest_energy, coordinates

In [ ]:

import os                                     
import re                                     
import pandas as pd                           
from typing import List, Tuple, Optional      

from rdkit import Chem                        
from rdkit.Chem import AllChem                
from rdkit.Chem import rdmolfiles             
from rdkit.Chem import AddHs, RWMol           
from rdkit.Geometry import Point3D            


try:
    from openbabel import openbabel, pybel                             
    _HAVE_PYBEL = True
except Exception:
    _HAVE_PYBEL = False


def _sanitize_filename(name: str) -> str:
    
    return re.sub(r'[\\/:*?"<>|\s]+', '_', str(name)).strip('_')


def _assign_coords_to_mol(mol: Chem.Mol, coordinates_list: List[Tuple[str, float, float, float]]) -> Chem.Mol:
    
    
    mol = Chem.Mol(mol)
    
    if mol.GetNumAtoms() != len(coordinates_list):
        raise ValueError("Public message removed for release.")

    
    conf = Chem.Conformer(mol.GetNumAtoms())
    for idx, (_sym, x, y, z) in enumerate(coordinates_list):
        conf.SetAtomPosition(idx, Point3D(x, y, z))

    
    mol.RemoveAllConformers()
    mol.AddConformer(conf, assignId=True)
    return mol


def _write_mol2(mol: Chem.Mol, out_path: str, title: str) -> None:
    
    
    if hasattr(rdmolfiles, "MolToMol2Block"):
        mol2_block = rdmolfiles.MolToMol2Block(mol)    
        
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(mol2_block)
        return

    
    if _HAVE_PYBEL:
        sdf_block = Chem.MolToMolBlock(mol)           
        obmol = pybel.readstring("sdf", sdf_block)    
        obmol.title = title                           
        obmol.write("mol2", out_path, overwrite=True) 
        return

    
    raise RuntimeError("Public message removed for release.")

In [ ]:



def _filter_valid_polymer_rows_for_topology(df: pd.DataFrame) -> pd.DataFrame:
    
    required_columns = {"Name", "SMILES"}
    if not required_columns.issubset(df.columns):
        raise ValueError("Public message removed for release.")

    valid_df = df.dropna(subset=["Name", "SMILES"]).copy()
    for column in ["Name", "SMILES"]:
        text_values = valid_df[column].astype(str).str.strip()
        valid_df = valid_df[(text_values != "") & (text_values.str.lower() != "nan")].copy()

    if valid_df.empty:
        raise ValueError("Public message removed for release.")

    return valid_df.reset_index(drop=True)

def create_monomer_mol2file(df: pd.DataFrame) -> None:
    
    
    if not {"Name", "SMILES"}.issubset(df.columns):
        raise ValueError("Public message removed for release.")

    
    valid_df = _filter_valid_polymer_rows_for_topology(df)

    
    for row in valid_df.itertuples(index=False):
        
        name = getattr(row, "Name")
        smiles = getattr(row, "SMILES")

        
        mol = Chem.MolFromSmiles(smiles)
        mol_with_h = AddHs(mol)

        
        terminal_indices = [atom.GetIdx() for atom in mol_with_h.GetAtoms() if atom.GetSymbol() == '*']

        
        rw_mol = RWMol(mol_with_h)

        
        for idx in terminal_indices:
            rw_mol.ReplaceAtom(idx, Chem.Atom(1))  
            rw_mol.GetAtomWithIdx(idx).SetNoImplicit(False)  

        
        rw_mol.UpdatePropertyCache(strict=False)

        
        new_mol = rw_mol.GetMol()

        
        new_mol_with_h = AddHs(new_mol)

        
        smiles_with_h = Chem.MolToSmiles(new_mol_with_h)

        
        lowest_conf_index, lowest_energy, coordinates_list = generate_lowest_energy_conformer_openbabel(smiles_with_h, num_confs=50, forcefield="UFF")
        

        
        if lowest_conf_index is None or coordinates_list is None:
            print("Public message removed for release.")
            continue

        
        try:
            mol3d = _assign_coords_to_mol(new_mol_with_h, coordinates_list)
        except Exception as e:
            print("Public message removed for release.")
            continue

        
        out_name = _sanitize_filename(name) + ".mol2"
        out_path = os.path.join(".", out_name)

        
        try:
            _write_mol2(mol3d, out_path, title=str(name))
            print("Public message removed for release.")
        except Exception as e:
            print("Public message removed for release.")


In [ ]:
_system_excel_file = "System_homopolymer.xlsx"
df = pd.read_excel(_system_excel_file)
df = _filter_valid_polymer_rows_for_topology(df)
df.to_excel(_system_excel_file, index=False)
create_monomer_mol2file(df) 

In [ ]:



def run_monomer_sobtop(mol2_path, fchk_path, top_path, itp_path):
    
    
    commands = f'1\n2\n7\n{fchk_path}\n{top_path}\n{itp_path}\n0\n'
    
    
    subprocess.run(['cd', sobtop_directory], shell=True)
    
    
    
    process = subprocess.Popen(
        ['./sobtop', mol2_path],
        cwd=sobtop_directory,
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    
    stdout, stderr = process.communicate(commands)

    
    print(stdout)
    
    
    if stderr:
        print(stderr)

In [ ]:


def create_monomer_itp_top(df):
    for name in df["Name"]:
        
        mol2_path = os.path.abspath(f"{name}" + '.mol2')
        fchk_path = os.path.abspath(f"{name}" + '.fchk')
        top_path = os.path.abspath(f"{name}" + '.top')
        itp_path = os.path.abspath(f"{name}" + '.itp')
        print(f"!!!!!!!!!!!!!!!!!!!!!!!!!!!!!{itp_path}!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
        
        run_monomer_sobtop(mol2_path, fchk_path, top_path, itp_path)

In [ ]:

create_monomer_itp_top(df)

In [ ]:


import os                             
import re                             
from typing import Dict, Tuple        


def _split_data_and_comment(line: str) -> Tuple[str, str]:
    
    pos = line.find(';')                          
    if pos == -1:                                 
        return line.rstrip('\n'), ""              
    return line[:pos].rstrip(), line[pos:].rstrip('\n')  

def _leading_ws(line: str) -> str:
    
    m = re.match(r'^(\s*)', line)                 
    return m.group(1) if m else ""                

def _is_section_header(line: str, name: str) -> bool:
    
    pat = rf'^\s*\[\s*{re.escape(name)}\s*\]\s*$' 
    return re.match(pat, line, flags=re.IGNORECASE) is not None

def _find_section_bounds(lines):
    
    headers = []                                   
    for i, ln in enumerate(lines):                 
        m = re.match(r'^\s*\[\s*([A-Za-z0-9_]+)\s*\]\s*.*$', ln)  
        if m:
            headers.append((m.group(1).lower(), i))          
    bounds: Dict[str, Tuple[int, int]] = {}        
    for k, (sec_name, start_idx) in enumerate(headers):
        next_idx = headers[k + 1][1] if k + 1 < len(headers) else len(lines)  
        bounds[sec_name] = (start_idx + 1, next_idx)  
    return bounds

def _int_token(s: str) -> bool:
    
    return re.fullmatch(r'[+-]?\d+', s) is not None


def generate_fake_itp_from_atom_list(df) -> None:
    

    
    if "Name" not in df.columns:                       
        raise ValueError("Public message removed for release.")

    
    for name in df["Name"]:
        name = str(name)                               
        try:
            
            atom_list = load_polymer_atom_list(f"{name}_atom_list.json")  
            if not isinstance(atom_list, list):        
                raise TypeError("Public message removed for release.")

            
            itp_path = os.path.join(".", f"{name}.itp")  
            if not os.path.isfile(itp_path):             
                print("Public message removed for release.")
                continue
            with open(itp_path, "r", encoding="utf-8") as f:
                lines = f.readlines()                    

            
            bounds = _find_section_bounds(lines)         
            if "atoms" not in bounds:
                raise ValueError("Public message removed for release.")
            atoms_s, atoms_e = bounds["atoms"]           
            bonds_s, bonds_e = bounds.get("bonds", (None, None))   
            angles_s, angles_e = bounds.get("angles", (None, None))

            
            idx_to_type_new: Dict[int, str] = {}         
            new_atom_lines = []                           
            type_count = 0                                

            for li in range(atoms_s, atoms_e):
                raw = lines[li]                           
                stripped = raw.strip()                    
                if stripped == "" or stripped.startswith(";"):
                    new_atom_lines.append(raw)            
                    continue

                data, cmt = _split_data_and_comment(raw)  
                toks = data.split()                       
                if len(toks) < 2:
                    
                    new_atom_lines.append(raw)
                    print("Public message removed for release.")
                    continue

                
                try:
                    idx_val = int(toks[0])                
                except ValueError:
                    new_atom_lines.append(raw)            
                    print("Public message removed for release.")
                    continue

                
                if type_count >= len(atom_list):
                    raise ValueError("Public message removed for release.")
                new_type = str(atom_list[type_count])     
                toks[1] = new_type                        
                type_count += 1                           

                idx_to_type_new[idx_val] = new_type       

                
                lead = _leading_ws(raw)                   
                rebuilt = lead + " ".join(toks)           
                if cmt:                                   
                    if not rebuilt.endswith(" "):
                        rebuilt += "  "                   
                    rebuilt += cmt
                if not rebuilt.endswith("\n"):            
                    rebuilt += "\n"
                new_atom_lines.append(rebuilt)            

            
            if type_count != len(atom_list):
                raise ValueError("Public message removed for release.")

            
            lines[atoms_s:atoms_e] = new_atom_lines

            
            if bonds_s is not None and bonds_e is not None:
                new_bond_lines = []                       
                for li in range(bonds_s, bonds_e):
                    raw = lines[li]
                    stripped = raw.strip()
                    if stripped == "" or stripped.startswith(";"):
                        new_bond_lines.append(raw)        
                        continue
                    data, cmt = _split_data_and_comment(raw)
                    toks = data.split()
                    if len(toks) < 2 or (not _int_token(toks[0])) or (not _int_token(toks[1])):
                        
                        new_bond_lines.append(raw)
                        continue
                    
                    i_idx = int(toks[0])
                    j_idx = int(toks[1])
                    
                    if i_idx not in idx_to_type_new or j_idx not in idx_to_type_new:
                        print("Public message removed for release.")
                        new_bond_lines.append(raw)
                        continue
                    toks[0] = idx_to_type_new[i_idx]
                    toks[1] = idx_to_type_new[j_idx]
                    lead = _leading_ws(raw)
                    rebuilt = lead + " ".join(toks)
                    if cmt:
                        if not rebuilt.endswith(" "):
                            rebuilt += "  "
                        rebuilt += cmt
                    if not rebuilt.endswith("\n"):
                        rebuilt += "\n"
                    new_bond_lines.append(rebuilt)
                lines[bonds_s:bonds_e] = new_bond_lines   

            
            if angles_s is not None and angles_e is not None:
                new_angle_lines = []                      
                for li in range(angles_s, angles_e):
                    raw = lines[li]
                    stripped = raw.strip()
                    if stripped == "" or stripped.startswith(";"):
                        new_angle_lines.append(raw)       
                        continue
                    data, cmt = _split_data_and_comment(raw)
                    toks = data.split()
                    if len(toks) < 3 or (not _int_token(toks[0])) or (not _int_token(toks[1])) or (not _int_token(toks[2])):
                        
                        new_angle_lines.append(raw)
                        continue
                    
                    i_idx = int(toks[0])
                    j_idx = int(toks[1])
                    k_idx = int(toks[2])
                    if (i_idx not in idx_to_type_new) or (j_idx not in idx_to_type_new) or (k_idx not in idx_to_type_new):
                        print("Public message removed for release.")
                        new_angle_lines.append(raw)
                        continue
                    toks[0] = idx_to_type_new[i_idx]
                    toks[1] = idx_to_type_new[j_idx]
                    toks[2] = idx_to_type_new[k_idx]
                    lead = _leading_ws(raw)
                    rebuilt = lead + " ".join(toks)
                    if cmt:
                        if not rebuilt.endswith(" "):
                            rebuilt += "  "
                        rebuilt += cmt
                    if not rebuilt.endswith("\n"):
                        rebuilt += "\n"
                    new_angle_lines.append(rebuilt)
                lines[angles_s:angles_e] = new_angle_lines 

            
            out_path = os.path.join(".", f"fake_{name}.itp")     
            with open(out_path, "w", encoding="utf-8") as f:
                f.writelines(lines)                              
            print("Public message removed for release.")

        except Exception as e:
            
            print("Public message removed for release.")
            continue

In [ ]:

generate_fake_itp_from_atom_list(df)

In [ ]:


def generate_fake_itp_for_copolymer(df, polymer_node_list_path: str = "polymer_node_list.json") -> None:
    
    import os  
    
    if "copolymer_name" not in df.columns:  
        raise ValueError("Public message removed for release.")
    if len(df) == 0:  
        raise ValueError("Public message removed for release.")

    
    copolymer_name = str(df["copolymer_name"].iloc[0])  
    if (copolymer_name is None) or (copolymer_name.strip() == "") or (copolymer_name.lower() == "nan"):
        raise ValueError("Public message removed for release.")

    
    atom_list = load_polymer_atom_list(polymer_node_list_path)  

    
    
    _suffix_num_pat = re.compile(r"_(\d+)$")  
    atom_list = [_suffix_num_pat.sub("", str(x)) for x in atom_list]  

    if not isinstance(atom_list, list):  
        raise TypeError("Public message removed for release.")

    
    itp_path = os.path.join(".", f"{copolymer_name}.itp")  
    if not os.path.isfile(itp_path):  
        raise FileNotFoundError("Public message removed for release.")
    with open(itp_path, "r", encoding="utf-8") as f:  
        lines = f.readlines()

    
    bounds = _find_section_bounds(lines)  
    if "atoms" not in bounds:  
        raise ValueError("Public message removed for release.")
    atoms_s, atoms_e = bounds["atoms"]  
    bonds_range = bounds.get("bonds", (None, None))  
    angles_range = bounds.get("angles", (None, None))  
    bonds_s, bonds_e = bonds_range  
    angles_s, angles_e = angles_range  

    
    type_entries = 0  
    for li in range(atoms_s, atoms_e):  
        raw = lines[li]  
        stripped = raw.strip()  
        if stripped == "" or stripped.startswith(";"):  
            continue
        data, _cmt = _split_data_and_comment(raw)  
        toks = data.split()  
        if len(toks) >= 2:  
            
            if _int_token(toks[0]):
                type_entries += 1  

    
    if type_entries != len(atom_list):  
        raise ValueError(
            "Public message removed for release."
        )

    
    idx_to_type_new = {}  
    new_atom_lines = []  
    take = 0  
    for li in range(atoms_s, atoms_e):  
        raw = lines[li]  
        stripped = raw.strip()  
        if stripped == "" or stripped.startswith(";"):  
            new_atom_lines.append(raw)  
            continue
        data, cmt = _split_data_and_comment(raw)  
        toks = data.split()  
        if len(toks) < 2 or (not _int_token(toks[0])):  
            new_atom_lines.append(raw)  
            print("Public message removed for release.")
            continue
        idx_val = int(toks[0])  
        new_type = str(atom_list[take])  
        toks[1] = new_type  
        idx_to_type_new[idx_val] = new_type  
        take += 1  
        lead = _leading_ws(raw)  
        rebuilt = lead + " ".join(toks)  
        if cmt:  
            if not rebuilt.endswith(" "):
                rebuilt += "  "  
            rebuilt += cmt  
        if not rebuilt.endswith("\n"):  
            rebuilt += "\n"
        new_atom_lines.append(rebuilt)  

    
    lines[atoms_s:atoms_e] = new_atom_lines  

    
    if (bonds_s is not None) and (bonds_e is not None):  
        new_bond_lines = []  
        for li in range(bonds_s, bonds_e):  
            raw = lines[li]  
            stripped = raw.strip()  
            if stripped == "" or stripped.startswith(";"):  
                new_bond_lines.append(raw)  
                continue
            data, cmt = _split_data_and_comment(raw)  
            toks = data.split()  
            
            if len(toks) >= 2 and _int_token(toks[0]) and _int_token(toks[1]):
                i_idx = int(toks[0])  
                j_idx = int(toks[1])  
                if (i_idx in idx_to_type_new) and (j_idx in idx_to_type_new):  
                    toks[0] = idx_to_type_new[i_idx]  
                    toks[1] = idx_to_type_new[j_idx]  
                    lead = _leading_ws(raw)  
                    rebuilt = lead + " ".join(toks)  
                    if cmt:  
                        if not rebuilt.endswith(" "):
                            rebuilt += "  "
                        rebuilt += cmt
                    if not rebuilt.endswith("\n"):
                        rebuilt += "\n"
                    new_bond_lines.append(rebuilt)  
                else:
                    print("Public message removed for release.")
                    new_bond_lines.append(raw)  
            else:
                new_bond_lines.append(raw)  
        lines[bonds_s:bonds_e] = new_bond_lines  

    
    if (angles_s is not None) and (angles_e is not None):  
        new_angle_lines = []  
        for li in range(angles_s, angles_e):  
            raw = lines[li]  
            stripped = raw.strip()  
            if stripped == "" or stripped.startswith(";"):  
                new_angle_lines.append(raw)  
                continue
            data, cmt = _split_data_and_comment(raw)  
            toks = data.split()  
            
            if len(toks) >= 3 and _int_token(toks[0]) and _int_token(toks[1]) and _int_token(toks[2]):
                i_idx = int(toks[0])  
                j_idx = int(toks[1])  
                k_idx = int(toks[2])  
                if (i_idx in idx_to_type_new) and (j_idx in idx_to_type_new) and (k_idx in idx_to_type_new):  
                    toks[0] = idx_to_type_new[i_idx]  
                    toks[1] = idx_to_type_new[j_idx]  
                    toks[2] = idx_to_type_new[k_idx]  
                    lead = _leading_ws(raw)  
                    rebuilt = lead + " ".join(toks)  
                    if cmt:  
                        if not rebuilt.endswith(" "):
                            rebuilt += "  "
                        rebuilt += cmt
                    if not rebuilt.endswith("\n"):
                        rebuilt += "\n"
                    new_angle_lines.append(rebuilt)  
                else:
                    print("Public message removed for release.")
                    new_angle_lines.append(raw)  
            else:
                new_angle_lines.append(raw)  
        lines[angles_s:angles_e] = new_angle_lines  

    
    out_path = os.path.join(".", f"fake_{copolymer_name}.itp")  
    with open(out_path, "w", encoding="utf-8") as f:  
        f.writelines(lines)  
    print("Public message removed for release.")  

In [ ]:

generate_fake_itp_for_copolymer(df, polymer_node_list_path="polymer_node_list.json")


In [ ]:

import os  

def build_monomer_bonds_angles(df):
    
    
    if "Name" not in df.columns:
        raise ValueError("Public message removed for release.")

    
    all_bonds = []   
    all_angles = []  

    
    for name in df["Name"]:
        
        itp_path = os.path.join(".", f"fake_{name}.itp")
        
        if not os.path.isfile(itp_path):
            print("Public message removed for release.")
            continue

        
        with open(itp_path, "r", encoding="utf-8") as f:
            lines = f.readlines()

        
        bounds = _find_section_bounds(lines)
        bonds_s, bonds_e = bounds.get("bonds", (None, None))
        angles_s, angles_e = bounds.get("angles", (None, None))

        
        if bonds_s is not None and bonds_e is not None:
            for li in range(bonds_s, bonds_e):
                raw = lines[li]                 
                stripped = raw.strip()          
                if stripped == "" or stripped.startswith(";"):
                    continue                    
                data, cmt = _split_data_and_comment(raw)  
                
                all_bonds.append((data.strip() + ("" if not cmt else "  " + cmt)).rstrip())

        
        if angles_s is not None and angles_e is not None:
            for li in range(angles_s, angles_e):
                raw = lines[li]                 
                stripped = raw.strip()          
                if stripped == "" or stripped.startswith(";"):
                    continue                    
                data, cmt = _split_data_and_comment(raw)  
                
                all_angles.append((data.strip() + ("" if not cmt else "  " + cmt)).rstrip())

    
    bonds_path = os.path.join(".", "monomer_bonds.txt")     
    angles_path = os.path.join(".", "monomer_angles.txt")   
    with open(bonds_path, "w", encoding="utf-8") as fb:
        for line in all_bonds:
            fb.write(line + "\n")
    with open(angles_path, "w", encoding="utf-8") as fa:
        for line in all_angles:
            fa.write(line + "\n")

    
    return bonds_path, angles_path

In [ ]:

bonds_path, angles_path = build_monomer_bonds_angles(df)

In [ ]:

def replace_fake_copolymer_from_monomers(fake_copolymer_itp_path):
    
    import os  

    
    base_name = os.path.basename(fake_copolymer_itp_path)         
    out_path = os.path.join(".", f"replace_{base_name}")          

    
    bonds_lib = os.path.join(".", "monomer_bonds.txt")
    angles_lib = os.path.join(".", "monomer_angles.txt")
    if not os.path.isfile(bonds_lib) or not os.path.isfile(angles_lib):
        raise FileNotFoundError("Public message removed for release.")

    
    bonds_map = {}  
    with open(bonds_lib, "r", encoding="utf-8") as fb:
        for raw in fb:
            line = raw.strip()                     
            if line == "" or line.startswith(";"):  
                continue
            data, cmt = _split_data_and_comment(line)  
            toks = data.split()                        
            if len(toks) < 3:                          
                continue
            k = (toks[0], toks[1])                     
            full_line = data + ("" if not cmt else "  " + cmt)  
            bonds_map[k] = full_line                   
            bonds_map[(k[1], k[0])] = full_line       

    
    angles_map = {}  
    with open(angles_lib, "r", encoding="utf-8") as fa:
        for raw in fa:
            line = raw.strip()                     
            if line == "" or line.startswith(";"):  
                continue
            data, cmt = _split_data_and_comment(line)  
            toks = data.split()                        
            if len(toks) < 4:                          
                continue
            k = (toks[0], toks[1], toks[2])            
            full_line = data + ("" if not cmt else "  " + cmt)  
            angles_map[k] = full_line                  

    
    with open(fake_copolymer_itp_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    
    bounds = _find_section_bounds(lines)
    bonds_s, bonds_e = bounds.get("bonds", (None, None))
    angles_s, angles_e = bounds.get("angles", (None, None))

    
    if bonds_s is not None and bonds_e is not None:
        new_bond_lines = []                                
        for li in range(bonds_s, bonds_e):
            raw = lines[li]                                
            stripped = raw.strip()                         
            if stripped == "" or stripped.startswith(";"): 
                new_bond_lines.append(raw)
                continue
            data, cmt_old = _split_data_and_comment(raw)   
            toks = data.split()                            
            if len(toks) < 2:                              
                new_bond_lines.append(raw)
                continue
            key = (toks[0], toks[1])                       
            if key in bonds_map:                           
                repl = bonds_map[key]                      
                lead = _leading_ws(raw)                    
                new_line = lead + repl                     
                if not new_line.endswith("\n"):
                    new_line += "\n"
                new_bond_lines.append(new_line)            
            else:
                new_bond_lines.append(raw)                 
        lines[bonds_s:bonds_e] = new_bond_lines            

    
    if angles_s is not None and angles_e is not None:
        new_angle_lines = []                               
        for li in range(angles_s, angles_e):
            raw = lines[li]                                
            stripped = raw.strip()                         
            if stripped == "" or stripped.startswith(";"): 
                new_angle_lines.append(raw)
                continue
            data, cmt_old = _split_data_and_comment(raw)   
            toks = data.split()                            
            if len(toks) < 3:                              
                new_angle_lines.append(raw)
                continue
            key = (toks[0], toks[1], toks[2])              
            if key in angles_map:                          
                repl = angles_map[key]                     
                lead = _leading_ws(raw)                    
                new_line = lead + repl                     
                if not new_line.endswith("\n"):
                    new_line += "\n"
                new_angle_lines.append(new_line)           
            else:
                new_angle_lines.append(raw)                
        lines[angles_s:angles_e] = new_angle_lines         

    
    with open(out_path, "w", encoding="utf-8") as fo:
        fo.writelines(lines)

    
    return out_path

In [ ]:
copolymer_name = df["copolymer_name"][0] 
out_path = replace_fake_copolymer_from_monomers(f"fake_{copolymer_name}.itp")

In [ ]:



def apply_calculated_params_to_copolymer(copolymer_itp_path, replace_fake_copolymer_itp_path):
    
    import os  

    
    with open(replace_fake_copolymer_itp_path, "r", encoding="utf-8") as fr:
        r_lines = fr.readlines()
    r_bounds = _find_section_bounds(r_lines)

    
    idx_to_type = {}  
    if "atoms" not in r_bounds:
        raise ValueError("Public message removed for release.")
    r_atoms_s, r_atoms_e = r_bounds["atoms"]
    for li in range(r_atoms_s, r_atoms_e):
        raw = r_lines[li]                                  
        stripped = raw.strip()                             
        if stripped == "" or stripped.startswith(";"):     
            continue
        data, _ = _split_data_and_comment(raw)             
        toks = data.split()                                
        if len(toks) >= 2 and _int_token(toks[0]):         
            idx_to_type[int(toks[0])] = toks[1]            

    
    r_bonds_s, r_bonds_e = r_bounds.get("bonds", (None, None))
    r_angles_s, r_angles_e = r_bounds.get("angles", (None, None))

    bonds_tail_map = {}  
    if r_bonds_s is not None and r_bonds_e is not None:
        for li in range(r_bonds_s, r_bonds_e):
            raw = r_lines[li]
            stripped = raw.strip()
            if stripped == "" or stripped.startswith(";"):
                continue
            data, cmt = _split_data_and_comment(raw)
            toks = data.split()
            if len(toks) < 3:  
                continue
            key = (toks[0], toks[1])                    
            tail_tokens = toks[2:]                      
            tail = " ".join(tail_tokens)                
            if cmt:
                tail += "  " + cmt                      
            bonds_tail_map[key] = tail                  
            bonds_tail_map[(key[1], key[0])] = tail     

    angles_tail_map = {}  
    if r_angles_s is not None and r_angles_e is not None:
        for li in range(r_angles_s, r_angles_e):
            raw = r_lines[li]
            stripped = raw.strip()
            if stripped == "" or stripped.startswith(";"):
                continue
            data, cmt = _split_data_and_comment(raw)
            toks = data.split()
            if len(toks) < 4:  
                continue
            key = (toks[0], toks[1], toks[2])           
            tail_tokens = toks[3:]                      
            tail = " ".join(tail_tokens)                
            if cmt:
                tail += "  " + cmt                      
            angles_tail_map[key] = tail                 

    
    with open(copolymer_itp_path, "r", encoding="utf-8") as fc:
        c_lines = fc.readlines()
    c_bounds = _find_section_bounds(c_lines)
    c_bonds_s, c_bonds_e = c_bounds.get("bonds", (None, None))
    c_angles_s, c_angles_e = c_bounds.get("angles", (None, None))

    
    if c_bonds_s is not None and c_bonds_e is not None:
        new_c_bonds = []                                   
        for li in range(c_bonds_s, c_bonds_e):
            raw = c_lines[li]                              
            stripped = raw.strip()                         
            if stripped == "" or stripped.startswith(";"): 
                new_c_bonds.append(raw)
                continue
            data, _orig_cmt = _split_data_and_comment(raw) 
            toks = data.split()
            if len(toks) < 2 or (not _int_token(toks[0])) or (not _int_token(toks[1])):
                new_c_bonds.append(raw)                    
                continue
            i_idx, j_idx = int(toks[0]), int(toks[1])      
            ti, tj = idx_to_type.get(i_idx), idx_to_type.get(j_idx)  
            if ti is None or tj is None:
                new_c_bonds.append(raw)                    
                continue
            tail = bonds_tail_map.get((ti, tj))            
            if tail is None:
                new_c_bonds.append(raw)                    
                continue
            
            lead = _leading_ws(raw)
            new_line = f"{lead}{toks[0]} {toks[1]} {tail}"
            if not new_line.endswith("\n"):
                new_line += "\n"
            new_c_bonds.append(new_line)
        c_lines[c_bonds_s:c_bonds_e] = new_c_bonds        

    
    if c_angles_s is not None and c_angles_e is not None:
        new_c_angles = []                                  
        for li in range(c_angles_s, c_angles_e):
            raw = c_lines[li]                              
            stripped = raw.strip()                         
            if stripped == "" or stripped.startswith(";"): 
                new_c_angles.append(raw)
                continue
            data, _orig_cmt = _split_data_and_comment(raw) 
            toks = data.split()
            
            if len(toks) < 3 or (not _int_token(toks[0])) or (not _int_token(toks[1])) or (not _int_token(toks[2])):
                new_c_angles.append(raw)
                continue
            i_idx, j_idx, k_idx = int(toks[0]), int(toks[1]), int(toks[2])  
            ti = idx_to_type.get(i_idx)                                      
            tj = idx_to_type.get(j_idx)
            tk = idx_to_type.get(k_idx)
            if (ti is None) or (tj is None) or (tk is None):
                new_c_angles.append(raw)                    
                continue
            tail = angles_tail_map.get((ti, tj, tk))        
            if tail is None:
                new_c_angles.append(raw)                    
                continue
            
            lead = _leading_ws(raw)
            new_line = f"{lead}{toks[0]} {toks[1]} {toks[2]} {tail}"
            if not new_line.endswith("\n"):
                new_line += "\n"
            new_c_angles.append(new_line)
        c_lines[c_angles_s:c_angles_e] = new_c_angles      

    
    out_path = os.path.join(".", f"calculated_{os.path.basename(copolymer_itp_path)}")  
    with open(out_path, "w", encoding="utf-8") as fo:
        fo.writelines(c_lines)

    return out_path  

In [ ]:
copolymer_name = df["copolymer_name"][0] 
new_out_path = apply_calculated_params_to_copolymer(copolymer_itp_path=f"{copolymer_name}.itp", 
                                                    replace_fake_copolymer_itp_path=f"replace_fake_{copolymer_name}.itp") 

In [ ]:

import os
import shutil
from typing import Tuple

def clean_files(copolymer_name: str,
                structure_extensions: Tuple[str] = ('.xyz', '.pdb', '.mol2'),
                topology_extensions: Tuple[str] = ('.top', '.itp', '.chg'),
                target_dir: str = "process_documentation") -> None:
    
    
    valid_extensions = structure_extensions + topology_extensions

    
    os.makedirs(target_dir, exist_ok=True)

    
    for file in os.listdir("."):
        if os.path.isfile(file):  
            _, ext = os.path.splitext(file)

            
            if ext in valid_extensions:
                
                valid_name1 = f"{copolymer_name}{ext}"
                
                valid_name2 = f"calculated_{copolymer_name}{ext}"

                
                if file not in (valid_name1, valid_name2):
                    shutil.move(file, os.path.join(target_dir, file))
                    print(f"Moved: {file} -> {target_dir}/")
                    
copolymer_name = df["copolymer_name"][0] 
clean_files(copolymer_name)

In [ ]:
import os
import shutil


os.makedirs('polymer_structure', exist_ok=True)
os.makedirs('polymer_topology', exist_ok=True)


def _move_file_with_replace(src_file: str, dst_dir: str) -> None:
    
    dst_file = os.path.join(dst_dir, os.path.basename(src_file))
    if os.path.isfile(dst_file) or os.path.islink(dst_file):
        os.remove(dst_file)
    elif os.path.isdir(dst_file):
        raise IsADirectoryError("Public message removed for release.")
    shutil.move(src_file, dst_dir)



for file in os.listdir('.'):
    if file.endswith(('.mol2', '.pdb', '.xyz')):
        _move_file_with_replace(file, 'polymer_structure')
    elif file.endswith(('.itp', '.top', '.chg')):
        _move_file_with_replace(file, 'polymer_topology')

In [ ]:
import os               
import shutil           
import zipfile          

def collect_and_compress_files(source_dir='.', cleanup=True):
    
    
    source_dir = os.path.abspath(source_dir)                              
    all_results_dir = os.path.join(source_dir, 'all_results')             
    zip_path = os.path.join(source_dir, 'all_results.zip')                
    targets = ['polymer_structure', 'polymer_topology']                              

    
    if os.path.isdir(all_results_dir):                                    
        shutil.rmtree(all_results_dir)
    os.makedirs(all_results_dir, exist_ok=True)                           
    if os.path.exists(zip_path):                                          
        os.remove(zip_path)

    
    for name in targets:                                                  
        src = os.path.join(source_dir, name)                              
        dst = os.path.join(all_results_dir, name)                         
        if os.path.isdir(src):                                            
            shutil.copytree(src, dst)                                     
        else:
            print("Public message removed for release.")                    

    
    with zipfile.ZipFile(zip_path, mode='w', compression=zipfile.ZIP_DEFLATED) as zf:  
        for root, _, files in os.walk(all_results_dir):                   
            for f in files:                                               
                fpath = os.path.join(root, f)                             
                arcname = os.path.relpath(fpath, all_results_dir)         
                zf.write(fpath, arcname)                                  

    
    if cleanup:                                                           
        shutil.rmtree(all_results_dir)

    print("Public message removed for release.")
    if cleanup:
        print("Public message removed for release.")


collect_and_compress_files(source_dir='.', cleanup=True)  